# 01 — Exploratory Data Analysis
**GeoAI Aquaculture Pond Identification Challenge** (Zindi / FAO / ITU)

Goal: understand the structure, class balance, missing values, and key signal in the raw data.

Data columns:
- `{band}_{t:02d}` for t ∈ 1..12 (12 monthly composites)
- SAR bands: `VH`, `VV`  (Sentinel-1, dB)
- Optical bands: `blue`, `green`, `red`, `re1`, `re2`, `re3`, `nir`, `nira`, `swir1`, `swir2`  (Sentinel-2, integer ×10000)
- Missing = -9999 (cloudy month or no data)
- `label`: 1 = aquaculture pond, 0 = other land cover

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_train, load_test, replace_missing, count_valid_timesteps, ALL_BANDS, N_TIMESTEPS

DATA = pathlib.Path('..') / 'data' / 'raw'
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 110

## 1. Load data

In [ ]:
train_raw, y = load_train(DATA / 'Train.csv')
test_raw     = load_test(DATA / 'Test.csv')

train = replace_missing(train_raw)
test  = replace_missing(test_raw)

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Positives: {y.sum()} ({y.mean()*100:.1f}%)  |  Negatives: {(~y.astype(bool)).sum()}')

## 2. Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
y.value_counts().plot.bar(ax=ax, color=['#4C72B0','#DD8452'])
ax.set_xticklabels(['Not pond (0)', 'Pond (1)'], rotation=0)
ax.set_ylabel('Count')
ax.set_title('Class distribution (train)')
plt.tight_layout(); plt.show()

## 3. Missing value analysis

In [ ]:
# Count missing per sample (train)
missing_per_sample = train.isna().sum(axis=1)
print(f'Avg missing features per train sample: {missing_per_sample.mean():.1f} / {train.shape[1]}')
print(f'Avg missing features per test sample:  {test.isna().sum(axis=1).mean():.1f} / {test.shape[1]}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
missing_per_sample.hist(bins=30, ax=axes[0], color='#4C72B0')
axes[0].set_title('Train: missing features per sample')
axes[0].set_xlabel('# missing')

test.isna().sum(axis=1).hist(bins=30, ax=axes[1], color='#DD8452')
axes[1].set_title('Test: missing features per sample')
axes[1].set_xlabel('# missing')
plt.tight_layout(); plt.show()

In [ ]:
# Missing by time step
vh_cols = [f'VH_{t:02d}' for t in range(1, N_TIMESTEPS + 1)]
opt_cols = [f'nir_{t:02d}' for t in range(1, N_TIMESTEPS + 1)]

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
missing_sar = train[vh_cols].isna().mean() * 100
missing_opt = train[opt_cols].isna().mean() * 100

missing_sar.plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('% missing VH per time step (train)')
axes[0].set_xlabel('Month')
axes[0].set_xticklabels(range(1, 13), rotation=0)

missing_opt.plot.bar(ax=axes[1], color='coral')
axes[1].set_title('% missing NIR per time step (train)')
axes[1].set_xlabel('Month')
axes[1].set_xticklabels(range(1, 13), rotation=0)
plt.tight_layout(); plt.show()

## 4. Band distributions by class

In [ ]:
# Compare mean band value over time steps, by class
key_bands = ['VH', 'VV', 'nir', 'green', 'swir1', 'red']

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for ax, band in zip(axes, key_bands):
    cols = [f'{band}_{t:02d}' for t in range(1, N_TIMESTEPS + 1)]
    pond_mean    = train.loc[y == 1, cols].mean(axis=0).values
    nopond_mean  = train.loc[y == 0, cols].mean(axis=0).values
    months = range(1, N_TIMESTEPS + 1)
    ax.plot(months, pond_mean,   label='Pond (1)',     color='#DD8452', lw=2)
    ax.plot(months, nopond_mean, label='No pond (0)',  color='#4C72B0', lw=2)
    ax.set_title(band)
    ax.set_xlabel('Month')
    ax.legend(fontsize=8)

plt.suptitle('Mean band value by month and class', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 5. SAR backscatter — pond vs no-pond

In [ ]:
vh_mean = train[[f'VH_{t:02d}' for t in range(1, N_TIMESTEPS+1)]].mean(axis=1)
vv_mean = train[[f'VV_{t:02d}' for t in range(1, N_TIMESTEPS+1)]].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for label, color, name in [(0, '#4C72B0', 'No pond'), (1, '#DD8452', 'Pond')]:
    mask = y == label
    axes[0].hist(vh_mean[mask].dropna(), bins=40, alpha=0.6, color=color, label=name, density=True)
    axes[1].hist(vv_mean[mask].dropna(), bins=40, alpha=0.6, color=color, label=name, density=True)

axes[0].set_title('Mean VH backscatter (dB)'); axes[0].legend()
axes[1].set_title('Mean VV backscatter (dB)'); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Optical water signal — NDWI preview

In [ ]:
EPS = 1e-6
green_mat = train[[f'green_{t:02d}' for t in range(1, N_TIMESTEPS+1)]].values.astype(float)
swir1_mat = train[[f'swir1_{t:02d}' for t in range(1, N_TIMESTEPS+1)]].values.astype(float)
ndwi_mat  = (green_mat - swir1_mat) / (green_mat + swir1_mat + EPS)

ndwi_mean = np.nanmean(ndwi_mat, axis=1)

fig, ax = plt.subplots(figsize=(6, 4))
for label, color, name in [(0, '#4C72B0', 'No pond'), (1, '#DD8452', 'Pond')]:
    mask = (y == label).values
    ax.hist(ndwi_mean[mask][~np.isnan(ndwi_mean[mask])], bins=50, alpha=0.65, color=color, label=name, density=True)
ax.axvline(0, color='k', linestyle='--', lw=1)
ax.set_title('Mean NDWI distribution by class')
ax.set_xlabel('NDWI (mean over 12 months)')
ax.legend()
plt.tight_layout(); plt.show()
print('Positive NDWI = more water-like signal')

## 7. Valid observation counts

In [ ]:
n_valid_train = count_valid_timesteps(train)
n_valid_test  = count_valid_timesteps(test)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
n_valid_train.value_counts().sort_index().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Train: # valid SAR months per sample')
n_valid_test.value_counts().sort_index().plot.bar(ax=axes[1], color='coral')
axes[1].set_title('Test: # valid SAR months per sample')
plt.tight_layout(); plt.show()

print('Test samples have fewer valid months (4-6) — model must be temporally robust')

## 8. Correlation between SAR and optical bands

In [ ]:
# Use monthly mean of each band as a summary
summary_cols = {}
for band in ALL_BANDS:
    cols = [f'{band}_{t:02d}' for t in range(1, N_TIMESTEPS+1)]
    summary_cols[band] = train[cols].mean(axis=1)

summary_df = pd.DataFrame(summary_cols)
summary_df['label'] = y

corr = summary_df.drop(columns=['label']).corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Mean band cross-correlation')
plt.tight_layout(); plt.show()

## Key Takeaways

1. **Class imbalance**: ~40% positives — F1-weighted metric handles this.
2. **SAR always available**: VH/VV present for all valid months; optical often missing (clouds).
3. **Test has fewer months**: models must aggregate across variable-length time series.
4. **NDWI discriminates well**: ponds show positive NDWI (green > SWIR1).
5. **SAR backscatter**: ponds tend to have lower (calmer water) VH/VV values.
6. **Temporal variation**: seasonal patterns differ between classes — temporal stats are key features.